In [1]:
import os
import zipfile
import pyarrow.parquet as pq
import pandas as pd

folder = r"F:\sanazi\python\data science\code\tamrin\proje\Tennis Schema\Tennis Schema\tennis_data"

data = {}

for file in os.listdir(folder): 
    if file.endswith(".zip"): 
        zip_path = os.path.join(folder, file) 

        with zipfile.ZipFile(zip_path) as z: 
            for parquet_file in z.namelist():
                if parquet_file.endswith(".parquet"): 

                    with z.open(parquet_file) as f: 
                        df = pq.read_table(f).to_pandas()
                    table_name =os.path.basename(os.path.dirname(parquet_file))[4:-8] + os.path.basename(parquet_file)[:-17] 

                    if table_name not in data:
                        data[table_name] = [] 
                    data[table_name].append(df) 


for table_name in data:
    data[table_name] = pd.concat(data[table_name], ignore_index=True)

print(data.keys())

C:\Users\sepehr\AppData\Local\Temp\ipykernel_19260\661795369.py:28: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  data[table_name] = pd.concat(data[table_name], ignore_index=True)


dict_keys(['matchaway_team', 'matchaway_team_score', 'matchevent', 'matchhome_team', 'matchhome_team_score', 'matchround', 'matchseason', 'matchtime', 'matchtournament', 'matchvenue', 'oddsodds', 'point_by_pointpbp', 'statisticsstatistics', 'tennis_powerpower', 'votesvotes'])


In [2]:
event = data["matchevent"]

print(event.columns)

Index(['match_id', 'first_to_serve', 'home_team_seed', 'away_team_seed',
       'custom_id', 'winner_code', 'default_period_count', 'start_datetime',
       'match_slug', 'final_result_only'],
      dtype='object')


In [3]:
rounds = data["matchround"]

print(rounds.columns)
print(rounds["name"].unique())

Index(['match_id', 'round_id', 'name', 'slug', 'cup_round_type'], dtype='object')
['Round of 16' 'Quarterfinal' 'Round of 32' 'Semifinal'
 'Qualification round 1' 'Final' 'Round of 64' 'Qualification round 2'
 'Semifinals' 'Quarterfinals' 'Round of 128']


In [ ]:
import pandas as pd
import numpy as np

event = data["matchevent"]
rounds = data["matchround"]
home = data["matchhome_team"][["match_id", "full_name"]].rename(
    columns={"full_name": "home_player"}
)
away = data["matchaway_team"][["match_id", "full_name"]].rename(
    columns={"full_name": "away_player"}
)


finals = event.merge(rounds, on="match_id")
finals = finals[finals["name"] == "Final"]


finals = finals.merge(home, on="match_id")
finals = finals.merge(away, on="match_id")


finals["winner"] = np.where(
    finals["winner_code"] == 1,
    finals["home_player"],
    finals["away_player"]
)


finals["month"] = pd.to_datetime(
    finals["start_datetime"], unit="s"
).dt.to_period("M")


wins = (
    finals.groupby(["month", "winner"])
    .size()
    .reset_index(name="tournaments_won")
)

print(wins)
best = wins.loc[wins["tournaments_won"].idxmax()]

print(best)

       month                   winner  tournaments_won
0    2024-02            Agwi, Michael               81
1    2024-02           Avdeeva, Julia               81
2    2024-02          Baez, Sebastian               16
3    2024-02            Banks, Amarni               16
4    2024-02  Barranco Cosano, Javier               16
..       ...                      ...              ...
227  2024-03         Zaytseva, Ksenia               16
228  2024-03        Zelníčková, Radka               24
229  2024-03          de Minaur, Alex               16
230  2024-03            van Wyk, Kris                8
231  2024-04         Podoroska, Nadia                1

[232 rows x 3 columns]
month                       2024-02
winner             Chidekh, Clement
tournaments_won                  97
Name: 13, dtype: object
